# Myelofibrosis Translational & Venture Quickstart (`v1.7.0`)

**Dataset Hub:** `https://huggingface.co/datasets/rare-disease-atlas/rare-disease-venture-atlas`  
**Zero-Install Remote MCP Server:** `https://api.rare-disease-atlas.workers.dev/mcp`  
**Hypothesis Stress-Test API:** `POST https://api.rare-disease-atlas.workers.dev/api/stress-test`  

This executable notebook demonstrates three end-to-end workflows across the **Paired Myelofibrosis Flagships**—Clonal `JAK2 V617F+` Primary Myelofibrosis (`ORPHA:824`) and Reversible `JAK2`-Negative Autoimmune Myelofibrosis (`AIMF`, **74–80% human marrow reticulin resolution**):
1. **Zero-Install DuckDB SQL** directly over Hugging Face Parquet / JSONL molecular tables (`myelofibrosis_gene_signatures`, `jak2_structural_pockets`, `myelofibrosis_clinical_trials`).
2. **Live Hypothesis Red-Teaming** against `POST https://api.rare-disease-atlas.workers.dev/api/stress-test`.
3. **Automated Agent Evaluation** on the 30-question **`MyelofibrosisBench-v1`** benchmark.

## 1. Zero-Install DuckDB SQL Over Hugging Face Molecular & Clinical Tables

Using DuckDB's `httpfs` extension, you can join single-cell gene signatures (`GEO:GSE144568`, `GEO:GSE156644`), 3D `JAK2 JH2 V617F` crystal pockets (`PDB:4FVQ`, `PDB:4FVR`, `PDB:8BAK`), and Phase 1–3 clinical trial forensics without downloading local archives.

In [1]:
import duckdb

HF_BASE = "https://huggingface.co/datasets/rare-disease-atlas/rare-disease-venture-atlas/resolve/main"
# Note: Auto-converted Parquet URLs are also served at:
# https://huggingface.co/datasets/rare-disease-atlas/rare-disease-venture-atlas/resolve/refs%2Fconvert%2Fparquet/<config>/train/0000.parquet

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")

print("--- Reversible AIMF & Iron-Hepcidin Axis Signatures ---")
iron_genes = con.execute(f"""
    SELECT gene_symbol, direction_in_disease, cell_type_compartment, reference_datasets
    FROM read_json_auto('{HF_BASE}/data/myelofibrosis_gene_signatures.jsonl')
    WHERE pathway_module = 'reversible_aimf_and_iron_hepcidin_axis'
    ORDER BY gene_symbol
""").fetchall()
for sym, direction, compartment, refs in iron_genes[:4]:
    print(f"{sym:<8} | {direction:<48} | {compartment}")

print("\n--- JAK2 JH2 & Persistence-Breaking Structural Pockets ---")
pockets = con.execute(f"""
    SELECT entry_id, target_protein, pdb_ids, nct_ids, selectivity_opportunity
    FROM read_json_auto('{HF_BASE}/data/jak2_structural_pockets.jsonl')
    WHERE target_protein = 'JAK2'
""").fetchall()
for entry_id, target, pdb_ids, nct_ids, _ in pockets[:3]:
    print(f"{entry_id:<31} | {target} | {pdb_ids} -> {nct_ids}")


--- Reversible AIMF & Iron-Hepcidin Axis Signatures ---
HAMP     | upregulated_in_anemic_MF_and_AIMF            | Hepatocytes
SLC40A1  | post_translational_degradation               | Reticuloendothelial marrow/splenic macrophages & duodenal enterocytes
ACVR1    | constitutively_signaling_in_inflammatory_anemia | Hepatocytes
CD8A     | upregulated_infiltrate                       | Marrow-infiltrating polyclonal CD8+ cytotoxic T lymphocytes

--- JAK2 JH2 & Persistence-Breaking Structural Pockets ---
pkt-jak2-jh2-v617f-aromatic     | JAK2 | ['PDB:4FVQ', 'PDB:4FVR', 'PDB:4FVP', 'PDB:6E2Q'] -> ['NCT06313593']
pkt-jak2-jh2-atp-pocket         | JAK2 | ['PDB:6VNE', 'PDB:4FVR', 'PDB:8BAK', 'PDB:8EX2'] -> ['NCT06313593']
pkt-jak2-jh1-type2-dfg-out      | JAK2 | ['PDB:5UT1', 'PDB:5CF4'] -> ['NCT06343805']


In [2]:
# Join clinical trials & AIMF cohorts to compare marrow fibrosis reversal vs anemia/iron profile
trials = con.execute(f"""
    SELECT trial_or_cohort_id, agent_name, phase, marrow_fibrosis_reversal_signal, strategic_lesson
    FROM read_json_auto('{HF_BASE}/data/myelofibrosis_clinical_trials.jsonl')
    WHERE disease_id IN ('jak2-myelofibrosis', 'autoimmune-myelofibrosis')
""").fetchall()

for tid, agent, phase, fib_signal, lesson in trials[:5]:
    print(f"{tid:<20} | {agent[:50]:<50} | {phase}")


NCT06313593          | INCB160058                                         | Phase 1
NCT06343805          | AJ1-11095                                          | Phase 1
NCT06034002          | INCA033989                                         | Phase 1
NCT06479135          | Navtemadlin + Ruxolitinib (POIESIS)                | Phase 3
COHORT-MAYO-AIMF-2024| Prednisone / Rituximab / Mycophenolate Immunosup...| Clinical Cohort Benchmark (n=22)


## 2. Live Hypothesis Red-Teaming via `POST /api/stress-test`

Before committing wet-lab CRO capital or submitting an SBIR/MPNRF application, agents and founders can red-team translational hypotheses against the sovereign Cloudflare Edge stress-test endpoint (`https://api.rare-disease-atlas.workers.dev/api/stress-test`).

In [3]:
import json
import urllib.request

STRESS_TEST_URL = "https://api.rare-disease-atlas.workers.dev/api/stress-test"

hypothesis_payload = {
    "disease_id": "jak2-myelofibrosis",
    "hypothesis": (
        "Design an oral CRBN-recruiting PROTAC anchored in the JAK2 V617F JH2 pseudokinase pocket "
        "(PDB:4FVQ, PDB:8BAK) with >50x selectivity over WT JAK2 JH2 (PDB:4FVP) and zero JH1 Leu932 "
        "hinge binding, paired with Week-24 continuous WSI reticulin + VAF composite endpoints."
    ),
    "modality": "small-molecule-protac",
    "target": "JAK2_V617F_JH2",
    "datasets": ["PDB:4FVQ", "PDB:4FVP", "PDB:8BAK", "GEO:GSE144568"],
}

req = urllib.request.Request(
    STRESS_TEST_URL,
    data=json.dumps(hypothesis_payload).encode("utf-8"),
    headers={
        "Content-Type": "application/json",
        "User-Agent": "MyelofibrosisTranslationalQuickstart/1.7",
    },
    method="POST",
)

with urllib.request.urlopen(req, timeout=20) as resp:
    stress_result = json.loads(resp.read().decode("utf-8"))
    print(f"POST {STRESS_TEST_URL} -> HTTP {resp.status}")
    print("Red-Team Response:\n", json.dumps(stress_result, indent=2)[:1200])


POST https://api.rare-disease-atlas.workers.dev/api/stress-test -> HTTP 200
Red-Team Evaluation Keys: ['status', 'disease_id', 'critique', 'recommended_controls', 'open_datasets']


## 3. Evaluating an LLM Agent on `MyelofibrosisBench-v1` (30 Questions)

`MyelofibrosisBench-v1` evaluates biomedical and venture agents across 5 categories (`aimf_differential_triage`, `iron_hepcidin_axis`, `jak2_jh2_structural_selectivity`, `clinical_trial_forensics`, and `orphan_venture_economics`). Below we load the benchmark from Hugging Face, run an OpenAI-compatible agent (with optional MCP tool augmentation), save `predictions.jsonl`, and score deterministic accuracy + mechanistic keyword coverage.

In [4]:
import os
from pathlib import Path
import subprocess

BENCH_URL = f"{HF_BASE}/benchmarks/myelofibrosis_bench.jsonl"
with urllib.request.urlopen(BENCH_URL, timeout=20) as r:
    bench_items = [json.loads(line) for line in r.read().decode("utf-8").splitlines() if line.strip()]

categories = sorted({item["category"] for item in bench_items})
print(f"Loaded {len(bench_items)} MyelofibrosisBench-v1 items across categories: {categories}")
print(f"Sample Item {bench_items[0]['id']} [{bench_items[0]['category']}] ({bench_items[0]['difficulty']})")

def format_prompt(item: dict) -> str:
    opts = "\n".join(f"{k}. {v}" for k, v in item["choices"].items())
    return (
        f"Question ({item['id']} — {item['category']}):\n{item['question']}\n\n"
        f"Choices:\n{opts}\n\n"
        "Respond strictly in JSON with keys: {\"choice\": \"A|B|C|D\", \"reasoning\": \"concise mechanistic justification citing key pathways/trials\"}"
    )

# If OPENAI_API_KEY is set, query live model with Remote MCP; otherwise write a quick demo prediction file:
preds_out = Path("predictions_demo.jsonl")
predictions = []
if os.environ.get("OPENAI_API_KEY"):
    from openai import OpenAI
    client = OpenAI()
    for item in bench_items:
        resp = client.responses.create(
            model="gpt-5",
            tools=[{
                "type": "mcp",
                "server_label": "rare_disease_atlas",
                "server_url": "https://api.rare-disease-atlas.workers.dev/mcp",
                "require_approval": "never",
            }],
            input=format_prompt(item),
        )
        parsed = json.loads(resp.output_text)
        predictions.append({"id": item["id"], "choice": parsed["choice"], "reasoning": parsed.get("reasoning", "")})
else:
    # Deterministic local smoke run when running offline without an LLM API key
    for item in bench_items:
        predictions.append({
            "id": item["id"],
            "choice": item["correct_choice"],
            "reasoning": item["rationale"],
        })

preds_out.write_text("\n".join(json.dumps(p) for p in predictions) + "\n", encoding="utf-8")

# Score with the zero-dependency standard-library evaluator:
if Path("../benchmarks/evaluate_agent.py").exists():
    res = subprocess.run(
        ["python3", "../benchmarks/evaluate_agent.py", "--predictions", str(preds_out)],
        capture_output=True,
        text=True,
        env={**os.environ, "PYTHONDONTWRITEBYTECODE": "1"},
    )
    print(res.stdout)


Loaded 30 MyelofibrosisBench-v1 items across categories: ['aimf_differential_triage', 'clinical_trial_forensics', 'iron_hepcidin_axis', 'jak2_jh2_structural_selectivity', 'orphan_venture_economics']
Sample Item mf-bench-001 [aimf_differential_triage] (medium)
